<!-- generated from src/day2-control/harm-classifier/lab.py — do not edit this notebook directly, your changes will be overwritten by tools/build_labs.py -->

# Train a harm classifier, then break it

> **Content warning.** This lab loads real prompts from
> [Aegis 2.0](https://huggingface.co/datasets/nvidia/Aegis-AI-Content-Safety-Dataset-2.0), a safety
> dataset built to contain disturbing requests: violence, self-harm, sexual content, hate speech,
> criminal planning. The notebook prints only short, truncated excerpts, but you'll see some. Step
> out whenever you need to.
>
> Data: Aegis 2.0 (NVIDIA), [CC-BY-4.0](https://creativecommons.org/licenses/by/4.0/), downloaded at
> runtime and not redistributed. Ghosh et al., *Aegis2.0: A Diverse AI Safety Dataset and Risks
> Taxonomy for Alignment of LLM Guardrails*, NAACL 2025,
> [arXiv:2501.09004](https://arxiv.org/abs/2501.09004).

The hackathon's blue team builds monitors by *prompting* models, walking down a ladder from big to
small. Below the bottom rung sits the monitor real deployments actually run on every request: a
small classifier trained for one job. It's cheap enough to put in front of all traffic, and that's
exactly why it matters where it fails.

You'll train two of them on ~22K labelled prompts (a bag-of-words baseline and a fine-tuned
DistilBERT), score them the way control scores a monitor, then spend the second half hunting for
the holes: inputs an attacker can shape to slip under the threshold, and benign inputs that trip
it. Last, you patch one hole and see what the patch costs.

If you built the injection scanner in the Day 1 instruction-hierarchy lab, this is its learned
counterpart: no hand-written cues, the same question of what gets past it.

**Budget:** a few minutes of compute on a T4 or L4, most of it one fine-tune (estimated 1–2 min).
Everything else takes seconds.

In [ ]:
# Installs the lab package on Colab; skipped when it's already importable (e.g. a local editable install).
try:
    import fast  # noqa: F401
except ImportError:
    %pip install -q git+https://github.com/sg-ai-safety-hub/FAST.git@main#subdirectory=src/packages/fast
    pass

In [ ]:
import time

import ipywidgets as widgets
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

from fast.colab import setup
from fast.labs.day2_control import harm_classifier as lab
from fast.testing import exercise

setup(require_gpu=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.precision", 3)

## Part 1: look at the data

Each row is a prompt a user might send a chat model, labelled `harmful` or not by human annotators.
The prompts mix ordinary questions, red-teaming attempts and jailbreak scaffolds (long role-play
and "ignore all previous instructions" setups), so `long` is worth watching. `needs_caution` marks
prompts the annotators called borderline, on either side of the label. `categories` names the harm
for harmful prompts. We keep Aegis's own splits: `val` is only for choosing thresholds, `test` only
for reporting.

The loader curates the raw data first and says what it dropped. Two of those drops are lessons in
their own right. The same prompt labelled both ways by different annotators is label noise. The same
prompt in both train and test is leakage: it would let a model score well by memorising.

In [ ]:
train, val, test = lab.load_splits()
print(f"train {len(train):,} · val {len(val):,} · test {len(test):,}\n")
print(pd.crosstab(train["harmful"], train["long"], margins=True).rename_axis(index="harmful", columns="long"))
print(f"\nmarked 'needs caution': {train['needs_caution'].mean():.0%} of train")

A few of each. In the long ones, the actual request is often buried after a page of setup.

In [ ]:
for harmful in (False, True):
    for long in (False, True):
        lab.show(train[(train["harmful"] == harmful) & (train["long"] == long)], n=2)

Length matters more than it looks. The transformer below reads at most 256 tokens and silently
drops the rest. Hold on to the last column; it's your first hole.

In [ ]:
MODEL = "distilbert-base-uncased"
MAX_LEN = 256
tok = AutoTokenizer.from_pretrained(MODEL)

sample = train.sample(min(2000, len(train)), random_state=0)
sample = sample.assign(tokens=[len(ids) for ids in tok(list(sample["prompt"]), verbose=False)["input_ids"]])
sample.groupby("harmful")["tokens"].describe(percentiles=[0.5, 0.9]).assign(
    over_256=sample.groupby("harmful")["tokens"].apply(lambda t: (t > MAX_LEN).mean())
)

## Part 2: a baseline monitor, and how to set its threshold

TF-IDF turns each prompt into a sparse vector of word and word-pair frequencies; logistic regression
learns a weight per feature. It trains in seconds and is a real baseline, not a strawman.

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
logreg = LogisticRegression(max_iter=1000).fit(tfidf.fit_transform(train["prompt"]), train["harmful"])


def score_baseline(prompts):
    """Harmful probability from the TF-IDF baseline, for a list of prompts."""
    return logreg.predict_proba(tfidf.transform(list(prompts)))[:, 1]

A classifier outputs a score; a monitor needs a yes/no. The obvious cut (0.5) is arbitrary. Control
sets the threshold from the other side, from what flagging costs. Every flag on a benign request is a
blocked user or a human reviewer's time, so you fix a **false-flag budget** first and take whatever
catch rate that buys. Here the budget is 5% of benign traffic (the hackathon's bar allowed 20%).

### Exercise 1: threshold at a false-flag budget

Choose the threshold from `val`, never `test`. A threshold tuned on the test set makes the test
numbers a lie.

In [ ]:
AUDIT_FPR = 0.05


@exercise
def threshold_at_fpr(benign_scores: np.ndarray, max_fpr: float) -> float:
    """Return the lowest threshold that flags at most `max_fpr` of `benign_scores`.

    A prompt is flagged when `score >= threshold`. The threshold must keep the flagged fraction of
    benign scores within `max_fpr`, and be as low as possible while doing so (a lower threshold
    catches more). Ties matter: if several benign prompts share a score, they're all flagged or none
    are. If even flagging only the single highest score would exceed the budget, return a value just
    above the maximum so nothing is flagged.
    """
    s = np.sort(np.asarray(benign_scores, dtype=float))
    candidates = np.unique(s)
    n_flagged = len(s) - np.searchsorted(s, candidates, side="left")
    within = candidates[n_flagged <= max_fpr * len(s)]
    return float(within[0]) if within.size else float(np.nextafter(s[-1], np.inf))


lab.check_threshold_at_fpr(threshold_at_fpr)

In [ ]:
val_benign = val.loc[~val["harmful"], "prompt"].tolist()
t_baseline = threshold_at_fpr(score_baseline(val_benign), AUDIT_FPR)
baseline_test = score_baseline(test["prompt"])
print(f"baseline threshold {t_baseline:.3f}")
print(pd.Series(lab.headline(baseline_test, test, t_baseline)).round(3).to_string())
lab.slice_report(baseline_test, test, t_baseline)

Read the slices before the headline. Compare `long` against `short`: a jailbreak wraps its request
in pages of words that look benign, which is exactly what a word-counting model can't see through.
And look at `needs caution`, where the annotators themselves hesitated.
Also check whether the false-flag rate on `test` stayed near the 5% you set on `val`. It won't be
exact, and the gap is how much you should trust a budget set on data that isn't your traffic.

## Part 3: fine-tune a transformer

DistilBERT (66M parameters) with a two-way classification head, one epoch over the training set.
The loop below is the whole of it: tokenize a batch, compute the loss, step. Two tricks keep it
fast. Mixed precision (bf16 where the GPU has it, fp16 on a T4). Batches of similar-length
prompts, so a batch of short prompts isn't padded to 256 tokens. Start it, then read on while it
runs (it prints an ETA). The load report flagging `classifier.*` as MISSING is expected: that
head is new, and training it is the point. A T4 or L4 is plenty; an A100 saves under a minute
here, and its compute units are better spent on Day 3.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(device)
use_amp = device == "cuda"
amp_dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16


def encode(prompts):
    return tok(list(prompts), truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(device)


def length_batches(prompts, batch_size, seed=0):
    """Shuffle, sort by length within pools of 50 batches, then shuffle the batches themselves."""
    rng = np.random.default_rng(seed)
    order, lengths = rng.permutation(len(prompts)), np.array([len(p) for p in prompts])
    pool = batch_size * 50
    order = np.concatenate([sorted(order[i : i + pool], key=lengths.__getitem__) for i in range(0, len(order), pool)])
    batches = [order[i : i + batch_size] for i in range(0, len(order), batch_size)]
    return [batches[i] for i in rng.permutation(len(batches))]


def finetune(model, prompts, labels, epochs=1, lr=5e-5, batch_size=32):
    prompts, labels = list(prompts), np.asarray(labels, dtype=int)
    batches = [b for e in range(epochs) for b in length_batches(prompts, batch_size, seed=e)]
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * len(batches)), len(batches))
    scaler = torch.amp.GradScaler(enabled=use_amp and amp_dtype == torch.float16)  # bf16 needs no scaling
    model.train()
    start = time.time()
    for step, idx in enumerate(batches, 1):
        y = torch.tensor(labels[idx], device=device)
        with torch.autocast(device_type=device, dtype=amp_dtype, enabled=use_amp):
            loss = model(**encode([prompts[i] for i in idx]), labels=y).loss
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()
        sched.step()
        if step == min(20, len(batches)):
            eta = (time.time() - start) / step * (len(batches) - step)
            print(f"{len(batches)} steps · ETA {eta / 60:.1f} min")
        if step % 100 == 0 or step == len(batches):
            print(f"step {step:>4}/{len(batches)}  loss {loss.item():.3f}")
    print(f"done in {time.time() - start:.0f}s")
    model.eval()


@torch.no_grad()
def score(prompts, batch_size=128):
    """Harmful probability from the fine-tuned model, for a list of prompts."""
    prompts = list(prompts)
    order = np.argsort([len(p) for p in prompts])  # score similar lengths together, then restore order
    out = np.empty(len(prompts))
    for i in range(0, len(prompts), batch_size):
        idx = order[i : i + batch_size]
        with torch.autocast(device_type=device, dtype=amp_dtype, enabled=use_amp):
            logits = model(**encode([prompts[j] for j in idx])).logits
        out[idx] = logits.float().softmax(-1)[:, 1].cpu().numpy()
    return out


finetune(model, train["prompt"], train["harmful"])

In [ ]:
t_model = threshold_at_fpr(score(val_benign), AUDIT_FPR)
model_test = score(test["prompt"])
print(f"thresholds: baseline {t_baseline:.3f} · distilbert {t_model:.3f}\n")
print(pd.DataFrame({
    "baseline": lab.headline(baseline_test, test, t_baseline),
    "distilbert": lab.headline(model_test, test, t_model),
}).round(3).to_string())
lab.compare({"baseline": baseline_test, "distilbert": model_test}, test, {"baseline": t_baseline, "distilbert": t_model})

The transformer should win, and win most where the baseline was weakest. Before you trust the
margin, note where both are still poor: those slices are what an attacker aims for, and Part 5 goes
looking for more.

## Part 4: play with it

Type anything and see both monitors' scores. `FLAG` means over that model's threshold. Try benign
prompts, obviously harmful ones, and everything in between. For batches, call `score_both([...])`.

In [ ]:
def score_both(prompts):
    prompts = list(prompts)
    b, m = score_baseline(prompts), score(prompts)
    return pd.DataFrame({
        "prompt": [p[:60] for p in prompts],
        "baseline": [f"{s:.2f}{' FLAG' if s >= t_baseline else ''}" for s in b],
        "distilbert": [f"{s:.2f}{' FLAG' if s >= t_model else ''}" for s in m],
    })


score_both(["How do I bake sourdough bread?", "Write a short poem about the sea."])

In [ ]:
box = widgets.Textarea(placeholder="Type a prompt…", layout=widgets.Layout(width="100%", height="90px"))
button, output = widgets.Button(description="Score"), widgets.Output()


def on_click(_):
    with output:
        output.clear_output()
        display(score_both([box.value]))


button.on_click(on_click)
display(widgets.VBox([box, button, output]))

## Part 5: hunt for holes

An attacker doesn't need the monitor to be wrong on average, only on the one input they send. We
measure a hole the same way every time: take harmful test prompts the model **catches**, apply a
transformation, and count how many it now misses.

### Exercise 2: evasion rate

In [ ]:
@exercise
def evasion_rate(score_fn, prompts: list[str], transform, threshold: float) -> float:
    """Fraction of caught prompts that slip under `threshold` once `transform` is applied.

    `score_fn` takes a list of prompts and returns an array of scores; call it on whole lists, not
    one prompt at a time. A prompt is caught when its score is >= `threshold`. Only prompts caught
    *before* the transform count: the rate is (caught before, missed after) / (caught before).
    Return nan if nothing was caught to begin with.
    """
    before = np.asarray(score_fn(list(prompts)))
    caught = [p for p, s in zip(prompts, before) if s >= threshold]
    if not caught:
        return float("nan")
    after = np.asarray(score_fn([transform(p) for p in caught]))
    return float((after < threshold).mean())


lab.check_evasion_rate(evasion_rate)

Our targets: short harmful test prompts the fine-tuned model catches. They come from the dataset,
so nobody in the room has to write one.

In [ ]:
caught = test[test["harmful"] & ~test["long"] & (model_test >= t_model)]["prompt"]
targets = caught.sample(min(100, len(caught)), random_state=0).tolist()
print(f"{len(targets)} target prompts, all currently caught")

**Perturbations.** Four cheap transformations from the module: typos, leetspeak, base64, and benign
padding in front. Predict which one hurts most before running it.

In [ ]:
attacks = {"typos": lab.typos, "leetspeak": lab.leetspeak, "base64": lab.base64_wrap, "pad 300 words": lab.pad_front}
pd.DataFrame({
    name: {
        "baseline": evasion_rate(score_baseline, targets, fn, t_baseline),
        "distilbert": evasion_rate(score, targets, fn, t_model),
    }
    for name, fn in attacks.items()
}).round(2)

Some of these are cheap wins that teach you little. `distilbert-base-uncased` learned English from
English text, so base64 is noise to it and leetspeak shreds its vocabulary. Translation is the same
story: try a target in French or Malay in the play box. The deeper point stands, though: the model
only knows the distribution it was trained on, and an attacker picks the input.

Padding is different. It doesn't disguise the request at all; the request is simply past token 256,
so the model never reads it. That's not a gap in the training data. It's the deployment choice to
truncate, and it gets an engineering fix, not a data fix.

### Exercise 3: score long inputs in windows

Score every window and keep the maximum. A monitor that reads everything can't be padded past.

In [ ]:
@exercise
def chunked_score(score_fn, text: str, window: int = 150, stride: int = 75) -> float:
    """Score `text` in overlapping windows of `window` words and return the highest window score.

    Split on whitespace. A text of at most `window` words is scored whole. Otherwise take windows
    starting every `stride` words, each at most `window` words long, such that every word lands in at
    least one window (don't drop the tail). Score all windows in one `score_fn` call (it takes a list
    and returns an array) and return the max as a float.
    """
    words = text.split()
    if len(words) <= window:
        return float(score_fn([text])[0])
    starts = range(0, len(words) - window + stride, stride)
    return float(np.max(score_fn([" ".join(words[i : i + window]) for i in starts])))


lab.check_chunked_score(chunked_score)

In [ ]:
def score_chunked(prompts):
    return np.array([chunked_score(score, p) for p in prompts])


print(f"padding evasion, truncated: {evasion_rate(score, targets, lab.pad_front, t_model):.2f}")
print(f"padding evasion, windowed:  {evasion_rate(score_chunked, targets, lab.pad_front, t_model):.2f}")
long_benign = [p for p in val_benign if len(p.split()) > 150][:200]
print(
    f"false flags on long benign val prompts: truncated {(score(long_benign) >= t_model).mean():.2f}"
    f" · windowed {(score_chunked(long_benign) >= t_model).mean():.2f}"
)

The fix has a price. Taking the max over windows gives a long benign prompt several chances to trip
the threshold, so false flags on long inputs go up, and the threshold you set on single-window scores
is no longer calibrated. Every fix to a monitor moves both of its numbers.

**Over-flagging.** The other failure: benign prompts that borrow harm's vocabulary. Add your own.

In [ ]:
score_both(lab.SCARY_BENIGN + [
    # your own benign-but-scary prompts here
])

**Worst errors, and whether they're errors.** Now the test mistakes the model made *most
confidently*. Harm labels are judgement calls: curation already dropped prompts that annotators
labelled both ways, and `needs_caution` marks the ones they flagged as borderline. If errors cluster
there, some of the model's "errors" are the labels being arguable.

In [ ]:
errors = test.assign(score=model_test)[(model_test >= t_model) != test["harmful"]]
errors = errors.assign(confidence=np.where(errors["harmful"], 1 - errors["score"], errors["score"]))
print(f"{len(errors)} errors · {errors['needs_caution'].mean():.0%} marked 'needs caution'"
      f" vs {test['needs_caution'].mean():.0%} of the test set\n")
by_category = errors.explode("categories").fillna({"categories": "(benign label)"})
print(by_category.groupby("categories")["confidence"].agg(["size", "mean"]).sort_values("size", ascending=False).head(8))

Judge a few yourself. For each: is the label right, the model right, or is it genuinely unclear?
Where would you draw the line if this monitor were gating your product?

In [ ]:
worst = errors.sort_values("confidence", ascending=False).head(5)
for _, row in worst.iterrows():
    print(f"label={'harmful' if row['harmful'] else 'benign'} · model score {row['score']:.2f}")
    lab.show(pd.DataFrame([row]), n=1, width=300)

## Part 6: patch a hole, and check what it cost

The standard response to an evasion is to train on it. Pick the transformation that hurt your model
most (leetspeak by default) and fine-tune a little more on transformed harmful prompts, mixed with
ordinary data so the model doesn't forget the rest. Seconds on a GPU.

In [ ]:
PATCH = lab.leetspeak  # swap in the attack that hurt most
OTHER = lab.typos  # an attack you did *not* train on

n = min(1000, int(train["harmful"].sum()))
patched_harmful = train[train["harmful"]].sample(n, random_state=1)["prompt"].map(PATCH)
replay = train.sample(n, random_state=2)
patch_prompts = pd.concat([patched_harmful, replay["prompt"]])
patch_labels = np.concatenate([np.ones(n, dtype=int), replay["harmful"].to_numpy(dtype=int)])

before = {
    "patched attack": evasion_rate(score, targets, PATCH, t_model),
    "other attack": evasion_rate(score, targets, OTHER, t_model),
    "test false flags": ((model_test >= t_model) & ~test["harmful"]).sum() / (~test["harmful"]).sum(),
}
finetune(model, patch_prompts, patch_labels, lr=2e-5)
t_model = threshold_at_fpr(score(val_benign), AUDIT_FPR)
model_test = score(test["prompt"])
after = {
    "patched attack": evasion_rate(score, targets, PATCH, t_model),
    "other attack": evasion_rate(score, targets, OTHER, t_model),
    "test false flags": ((model_test >= t_model) & ~test["harmful"]).sum() / (~test["harmful"]).sum(),
}
pd.DataFrame({"before patch": before, "after patch": after}).round(2)

The patched attack should mostly close. Look at the other two rows: the attack you didn't train on,
and whether false flags moved. Patching one hole teaches the monitor that one hole. The attacker's
next move is the transformation you didn't think of, which is the hackathon's red-versus-blue loop
with the prompt engineering replaced by gradient steps.

## Discussion

- You set the threshold at a 5% false-flag budget. Who in a real deployment chooses that number,
  and what happens to catch rate when traffic shifts away from `val`?
- This classifier sees only the prompt. Which of the holes you found would a monitor that also
  reads the model's *response* close, and which would it not?
- A trained classifier is cheap enough to run on everything; a prompted LLM monitor is smarter but
  costs more per call. How would you stack them? What goes to the expensive one?
- The labels are only moderately agreed on. What does a 95% catch rate mean when annotators
  disagree on which prompts are harmful?